In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

customers = spark.createDataFrame([
    (1,"John ","Hyderabad"),
    (2,"Alice","Chennai"),
    (3,None,"Bangalore")
],["customer_id","name","city"])

cars = spark.createDataFrame([
    (101,"Toyota","Camry",30000),
    (102,"Honda","Civic",-20000),
    (103,"Hyundai","i20",15000)
],["car_id","brand","model","price"])

sales = spark.createDataFrame([
    (1,1,101,"2024-01-01",1),
    (2,2,102,"2024-01-02",2),
    (3,99,103,"2024-01-03",1)
],["sale_id","customer_id","car_id","sale_date","quantity"])

sales = sales.withColumn("sale_date", to_date(col("sale_date")))

df = sales.join(customers,"customer_id","left").join(cars,"car_id")
df.display()


car_id,customer_id,sale_id,sale_date,quantity,name,city,brand,model,price
101,1,1,2024-01-01,1,John,Hyderabad,Toyota,Camry,30000
102,2,2,2024-01-02,2,Alice,Chennai,Honda,Civic,-20000
103,99,3,2024-01-03,1,null,null,Hyundai,i20,15000


In [0]:
df=df.withColumn(
    'name',
    when(col('name').isNull(),'Unknown').otherwise(col("name"))
).withColumn(
    'city',
    when(col('city').isNull() ,'Unknown').otherwise(col('city'))
)
df.display()

car_id,customer_id,sale_id,sale_date,quantity,name,city,brand,model,price
101,1,1,2024-01-01,1,John,Hyderabad,Toyota,Camry,30000
102,2,2,2024-01-02,2,Alice,Chennai,Honda,Civic,-20000
103,99,3,2024-01-03,1,Unknown,Unknown,Hyundai,i20,15000


In [0]:
df=df.filter(col('price')>0)
df.display()

car_id,customer_id,sale_id,sale_date,quantity,name,city,brand,model,price
101,1,1,2024-01-01,1,John,Hyderabad,Toyota,Camry,30000
103,99,3,2024-01-03,1,Unknown,Unknown,Hyundai,i20,15000


In [0]:

df1=sales.join(customers,'customer_id',"left_anti")
df2=sales.join(cars,"car_id","left_anti")
df2.display()
df1.display()

car_id,sale_id,customer_id,sale_date,quantity


customer_id,sale_id,car_id,sale_date,quantity
99,3,103,2024-01-03,1


In [0]:
from pyspark.sql.functions import *
result=df.agg(sum('price').alias('Total_cost'))
result.display()

Total_cost
45000


In [0]:
sales_count=df.agg(count("sale_id"))
sales_count.display()

count(sale_id)
2


In [0]:
df22=df.withColumn("Total_cost",col("price")*col("quantity"))\
    .groupBy("customer_id",'name')\
.agg(sum("Total_cost").alias("total"))

display(df22)

customer_id,name,total
1,John,30000
99,Unknown,15000


In [0]:
df_brand=df.groupBy('brand').count()
df_brand.display()

brand,count
Toyota,1
Hyundai,1


In [0]:
print(df.columns)

['car_id', 'customer_id', 'sale_id', 'sale_date', 'quantity', 'name', 'city', 'brand', 'model', 'price']


In [0]:

df_city=df.groupBy('city').agg(sum("price")*count("city"))

df_city.display()


city,(sum(price) * count(city))
Hyderabad,30000
Unknown,15000


In [0]:
# data=df.groupBy(year(col("sale_date")),month(col("sale_date")))\
#     .agg(sum(col("price") * col('quantity')).alias("Total_price"))
# data.display()


data = df.groupBy(
    year(col("sale_date")).alias("year"),
    month(col("sale_date")).alias("month")
).agg(
    sum(col("price") * col("quantity")).alias("monthly_revenue")
).orderBy("year", "month")

data.display()

year,month,monthly_revenue
2024,1,45000
